In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

In [4]:
DATA_DIR = "data/"

# ── Weekly fruit & veg price ──────────────────────────────────────────────────
df_fruitveg = pd.read_csv(f"{DATA_DIR}cleaned_fruit_veg.csv")
df_fruitveg["date"] = pd.to_datetime(df_fruitveg["date"])
df_fruitveg["price"] = pd.to_numeric(df_fruitveg["price"], errors="coerce")
df_fruitveg = df_fruitveg[["date", "price"]].sort_values("date")

price_weekly = (
    df_fruitveg
    .groupby("date", as_index=False)["price"]
    .mean()
    .dropna()
    .sort_values("date")
    .set_index("date")["price"]
)

print(f"Price weekly: {len(price_weekly)} rows | {price_weekly.index[0].date()} → {price_weekly.index[-1].date()}")

# ── Weekly fuel prices ────────────────────────────────────────────────────────
fuel1 = pd.read_csv(f"{DATA_DIR}weekly_road_fuel_prices_2003_to_2017.csv")
fuel2 = pd.read_csv(f"{DATA_DIR}weekly_road_fuel_prices_2018_to_now.csv")
df_fuel = pd.concat([fuel1, fuel2], ignore_index=True)

# Find Date column and ULSP column
date_col  = [c for c in df_fuel.columns if c.strip().lower() == "date"][0]
ulsp_col  = [c for c in df_fuel.columns if "ulsp" in c.lower()][0]

df_fuel["date"]       = pd.to_datetime(df_fuel[date_col], dayfirst=True)
df_fuel["fuel_price"] = pd.to_numeric(df_fuel[ulsp_col], errors="coerce")
df_fuel = df_fuel[["date", "fuel_price"]].sort_values("date").dropna()
df_fuel = df_fuel[df_fuel["date"] >= price_weekly.index.min()]

fuel_weekly = (
    df_fuel
    .groupby("date", as_index=False)["fuel_price"]
    .mean()
    .set_index("date")["fuel_price"]
)

print(f"Fuel weekly:  {len(fuel_weekly)} rows | {fuel_weekly.index[0].date()} → {fuel_weekly.index[-1].date()}")

Price weekly: 527 rows | 2015-01-09 → 2026-02-16
Fuel weekly:  580 rows | 2015-01-12 → 2026-02-16


In [7]:
# ── Monthly CPI ───────────────────────────────────────────────────────────────
raw_cpi = pd.read_excel(
    f"{DATA_DIR}consumerpriceinflationdetailedreferencetables.xlsx",
    sheet_name="Table 15a, 15b, 15c",
    skiprows=5,
    header=0
)

# Strip leading/trailing spaces from all column names
raw_cpi.columns = raw_cpi.columns.str.strip()

months   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
year_col = "Unnamed: 1"

cpi_rows = raw_cpi[raw_cpi[year_col].astype(str).str.match(r"^\d{4}$", na=False)].copy()
cpi_rows = cpi_rows[[year_col] + months].copy()
cpi_rows.columns = ["year"] + months
cpi_rows["year"] = cpi_rows["year"].astype(int)

cpi_long = cpi_rows.melt(id_vars="year", value_vars=months, var_name="month", value_name="cpi")
cpi_long["cpi"] = pd.to_numeric(
    cpi_long["cpi"].astype(str).str.replace("..", "", regex=False).str.strip(),
    errors="coerce"
)
cpi_long["date"] = pd.to_datetime(
    cpi_long["year"].astype(str) + " " + cpi_long["month"] + " 01",
    format="%Y %b %d"
)
cpi_monthly = cpi_long.dropna(subset=["cpi"]).set_index("date")["cpi"].sort_index()

print(f"CPI monthly: {len(cpi_monthly)} rows | {cpi_monthly.index[0].date()} → {cpi_monthly.index[-1].date()}")

CPI monthly: 944 rows | 1998-01-01 → 2026-01-01


In [8]:
raw_api = pd.read_csv(f"{DATA_DIR}API_20260129.csv")
raw_api["date"] = pd.to_datetime(raw_api["date"])
raw_api["api"]  = pd.to_numeric(raw_api["index"], errors="coerce")
api_monthly = (
    raw_api[["date","api"]].dropna()
    .set_index("date")["api"]
    .sort_index()
)
print(f"API monthly:  {len(api_monthly)} rows | {api_monthly.index[0].date()} → {api_monthly.index[-1].date()}")

API monthly:  14981 rows | 2014-01-01 → 2025-11-01


In [10]:
raw_fert = pd.read_excel(
    f"{DATA_DIR}GBFertiliserPriceSeries.xlsx",
    sheet_name="GB Fertiliser Price Series ",
    skiprows=13,
    dtype=str
)

print("Fert columns:", raw_fert.columns.tolist())
print("\nFert first 5 rows:")
print(raw_fert.iloc[:5, :6])

Fert columns: ['Unnamed: 0', 'Month', 'AN – UK produced (34.5% N)', 'Unnamed: 3', 'AN – imported* \n(34.5% N)', 'Unnamed: 5', 'Granular Urea - standard specification (46% N)', 'Unnamed: 7', 'UAN (30% N w/w kg per 100kg)', 'Unnamed: 9', 'Muriate of Potash (MOP)', 'Unnamed: 11', 'Diammonium Phosphate (DAP)', 'Unnamed: 13', 'Triple Super Phosphate (TSP)', 'Unnamed: 15', 'Polysulphate', 'Unnamed: 17', 'Nitrate Sulphur (26N + 30-37SO3)', 'Unnamed: 19']

Fert first 5 rows:
  Unnamed: 0                Month AN – UK produced (34.5% N) Unnamed: 3  \
0        NaN  2017-01-01 00:00:00         239.27272727272728          →   
1        NaN  2017-02-01 00:00:00                      248.5          →   
2        NaN  2017-03-01 00:00:00         244.46666666666667          ↘   
3        NaN  2017-04-01 00:00:00         237.66666666666666          →   
4        NaN  2017-05-01 00:00:00         209.20833333333334          ↓   

  AN – imported* \n(34.5% N) Unnamed: 5  
0         232.57142857142858       

In [11]:
raw_fert = pd.read_excel(
    f"{DATA_DIR}GBFertiliserPriceSeries.xlsx",
    sheet_name="GB Fertiliser Price Series ",
    skiprows=13,
    dtype=str
)

an_col = "AN – UK produced (34.5% N)"

raw_fert["date"]       = pd.to_datetime(raw_fert["Month"], errors="coerce")
raw_fert["fertiliser"] = pd.to_numeric(raw_fert[an_col], errors="coerce")

fert_monthly = (
    raw_fert[["date","fertiliser"]].dropna()
    .set_index("date")["fertiliser"]
    .sort_index()
)

print(f"Fert monthly: {len(fert_monthly)} rows | {fert_monthly.index[0].date()} → {fert_monthly.index[-1].date()}")

Fert monthly: 93 rows | 2017-01-01 → 2026-01-01


In [13]:
raw_sppi = pd.read_csv(
    f"{DATA_DIR}series-210226.csv",
    names=["period","value"]
)
raw_sppi = raw_sppi[raw_sppi["period"].str.match(r"^\d{4} Q[1-4]$", na=False)].copy()
raw_sppi["sppi"] = pd.to_numeric(raw_sppi["value"], errors="coerce")

# Parse "YYYY QN" manually
def quarter_to_date(s):
    year, q = s.split(" Q")
    month = (int(q) - 1) * 3 + 1
    return pd.Timestamp(year=int(year), month=month, day=1)

raw_sppi["date"] = raw_sppi["period"].apply(quarter_to_date)

sppi_quarterly = (
    raw_sppi[["date","sppi"]].dropna()
    .set_index("date")["sppi"]
    .sort_index()
)
print(f"SPPI quarterly: {len(sppi_quarterly)} rows | {sppi_quarterly.index[0].date()} → {sppi_quarterly.index[-1].date()}")

SPPI quarterly: 119 rows | 1996-01-01 → 2025-07-01


In [16]:
weekly_idx = price_weekly.index

def locf_to_weekly(series, weekly_index):
    # Deduplicate by taking the mean for any duplicate dates
    series = series.groupby(series.index).mean()
    combined = series.reindex(series.index.union(weekly_index))
    combined = combined.ffill()
    return combined.reindex(weekly_index)

cpi_w  = locf_to_weekly(cpi_monthly,    weekly_idx)
api_w  = locf_to_weekly(api_monthly,    weekly_idx)
fert_w = locf_to_weekly(fert_monthly,   weekly_idx)
sppi_w = locf_to_weekly(sppi_quarterly, weekly_idx)
fuel_w = locf_to_weekly(fuel_weekly,    weekly_idx)

df = pd.DataFrame({
    "price":      price_weekly.values,
    "fuel_price": fuel_w.values,
    "cpi":        cpi_w.values,
    "api":        api_w.values,
    "fertiliser": fert_w.values,
    "sppi":       sppi_w.values,
}, index=weekly_idx).dropna()

print(f"\nAligned dataset: {df.shape} | {df.index[0].date()} → {df.index[-1].date()}")


Aligned dataset: (427, 6) | 2017-01-06 → 2026-02-16


In [17]:
scaler       = StandardScaler()
price_scaler = StandardScaler()

data_scaled = scaler.fit_transform(df.values)
price_scaler.fit(df[["price"]].values)

SEQ_LEN  = 52
PRED_LEN = 1

def make_sequences(data, seq_len, pred_len):
    X, y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len : i + seq_len + pred_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = make_sequences(data_scaled, SEQ_LEN, PRED_LEN)
print(f"Sequences: X={X.shape}, y={y.shape}")

n       = len(X)
n_train = int(n * 0.70)
n_val   = int(n * 0.85)

X_train, y_train = X[:n_train],       y[:n_train]
X_val,   y_val   = X[n_train:n_val],  y[n_train:n_val]
X_test,  y_test  = X[n_val:],         y[n_val:]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_val_t   = torch.FloatTensor(X_val)
y_val_t   = torch.FloatTensor(y_val)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32, shuffle=True
)

Sequences: X=(375, 52, 6), y=(375, 1)
Train: 262 | Val: 56 | Test: 57


In [ ]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = int(expand * d_model)

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d  = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner, bias=True
        )
        self.act     = nn.SiLU()
        self.x_proj  = nn.Linear(self.d_inner, d_state * 2 + self.d_inner, bias=False)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(self.d_inner, -1)
        self.A_log   = nn.Parameter(torch.log(A))
        self.D       = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj= nn.Linear(self.d_inner, d_model, bias=False)
        self.norm    = nn.LayerNorm(d_model)

    def ssm(self, x):
        B, L, d = x.shape
        A = -torch.exp(self.A_log)
        x_proj = self.x_proj(x)
        delta, B_mat, C = torch.split(
            x_proj, [self.d_inner, self.d_state, self.d_state], dim=-1
        )
        delta = nn.functional.softplus(self.dt_proj(delta))  # ← fixed
        dA    = torch.exp(torch.einsum("bld,dn->bldn", delta, A))
        dB    = torch.einsum("bld,bln->bldn", delta, B_mat)

        h  = torch.zeros(B, d, self.d_state, device=x.device)
        ys = []
        for i in range(L):
            h  = dA[:, i] * h + dB[:, i] * x[:, i, :].unsqueeze(-1)
            yi = torch.einsum("bdn,bn->bd", h, C[:, i])
            ys.append(yi)
        y = torch.stack(ys, dim=1)
        return y + x * self.D

    def forward(self, x):
        residual = x
        x  = self.norm(x)
        xz = self.in_proj(x)
        x_proj, z = xz.chunk(2, dim=-1)
        x_proj = self.conv1d(x_proj.transpose(1,2))[..., :x.shape[1]].transpose(1,2)
        x_proj = self.act(x_proj)
        y = self.ssm(x_proj)
        y = y * self.act(z)
        return self.out_proj(y) + residual


class MambaForecast(nn.Module):
    def __init__(self, input_dim, d_model=64, n_layers=2,
                 d_state=16, pred_len=1, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.layers     = nn.ModuleList([
            MambaBlock(d_model, d_state=d_state) for _ in range(n_layers)
        ])
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(d_model, pred_len)

    def forward(self, x):
        x = self.input_proj(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = self.dropout(x)
        return self.head(x[:, -1, :])

In [19]:
model = MambaForecast(
    input_dim = X_train.shape[2],
    d_model   = 64,
    n_layers  = 2,
    d_state   = 16,
    pred_len  = PRED_LEN,
    dropout   = 0.1
)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion = nn.MSELoss()

EPOCHS    = 150
best_val  = float("inf")
patience  = 20
wait      = 0
train_losses, val_losses = [], []
best_state = None

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())

    train_loss = np.mean(batch_losses)
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t).item()

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

model.load_state_dict(best_state)
print(f"Best val loss: {best_val:.4f}")


Parameters: 129,665


AttributeError: module 'torch' has no attribute 'softplus'

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).numpy()
    test_true = y_test_t.numpy()

test_pred_inv = price_scaler.inverse_transform(test_pred)
test_true_inv = price_scaler.inverse_transform(test_true)

mae  = np.mean(np.abs(test_pred_inv - test_true_inv))
rmse = np.sqrt(np.mean((test_pred_inv - test_true_inv) ** 2))
mape = np.mean(np.abs((test_true_inv - test_pred_inv) / (np.abs(test_true_inv) + 1e-8))) * 100
naive_err = np.mean(np.abs(np.diff(df["price"].values)))
mase = mae / naive_err

print(f"\n══ TEST SET METRICS ══")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"MASE: {mase:.4f}")


In [ ]:
seq = torch.FloatTensor(data_scaled[-SEQ_LEN:]).unsqueeze(0)
forecasts = []

model.eval()
with torch.no_grad():
    for _ in range(52):
        pred = model(seq).item()
        forecasts.append(pred)
        new_step    = seq[0, -1, :].clone()
        new_step[0] = pred
        seq = torch.cat([seq[:, 1:, :], new_step.unsqueeze(0).unsqueeze(0)], dim=1)

fc_inv = price_scaler.inverse_transform(
    np.array(forecasts).reshape(-1, 1)
).flatten()

fc_dates = pd.date_range(df.index[-1] + pd.Timedelta(weeks=1), periods=52, freq="W")

print(f"\n══ 52-WEEK FORECAST SUMMARY ══")
print(f"Mean: {fc_inv.mean():.4f} | Min: {fc_inv.min():.4f} | Max: {fc_inv.max():.4f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14))

# Training curve
axes[0].plot(train_losses, label="Train", color="steelblue")
axes[0].plot(val_losses,   label="Validation", color="tomato")
axes[0].set_title("Mamba — Training & Validation Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Test: predicted vs actual
test_dates = df.index[-len(test_true_inv):]
axes[1].plot(test_dates, test_true_inv.flatten(),  label="Actual",    color="steelblue", linewidth=1.0)
axes[1].plot(test_dates, test_pred_inv.flatten(),  label="Predicted", color="tomato",    linewidth=1.0, linestyle="--")
axes[1].set_title("Mamba — Test Set: Predicted vs Actual", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Price Index")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 52-week forecast
hist_plot = df["price"].iloc[-104:]
axes[2].plot(hist_plot.index, hist_plot.values,  color="steelblue", linewidth=1.0, label="Historical")
axes[2].plot(fc_dates,        fc_inv,            color="tomato",    linewidth=1.2, linestyle="--", label="Forecast")
axes[2].axvline(df.index[-1], color="grey", linestyle=":", linewidth=0.8)
axes[2].set_title("Mamba — 52-Week Ahead Forecast", fontweight="bold")
axes[2].set_ylabel("Price Index")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("mamba_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: mamba_forecast.png")